# Silver Layer: Customers

**Goal:** take the raw `customers` table from Bronze and make a clean, trustworthy version of it in Silver.

Bronze holds the data exactly as it arrived, every column is text, nothing has been corrected. Silver is where it becomes usable.

Customers table transformation list
1. Look at the data first
2. Rename columns where the table name already makes the prefix redundant
3. Clean up city and state text formatting
4. Check for nulls in customers_id
5. Validate customers_id is unique
6. Drop _rescued_data (new — Bronze diagnostic column, not needed in Silver)
7. Write to Silver

### Step 1: Look at the data first

Before changing anything, I'll first inspect the data to see what is actually there.

In [0]:
# ============================================================
# SILVER: customers
# Reads the Bronze customers table, applies cleaning/casting,
# validates it, and writes it as a managed Delta table in Silver.
# ============================================================

from pyspark.sql import functions as F

In [0]:
# spark.table() opens a table that already exists in Unity Catalog.

df_customers_bronze = spark.table("olist_catalog.bronze.customers")

# display the bronze customet table
display(df_customers_bronze.limit(5))

### Step 2: Rename the columns

I'm dropping the customer_ prefix in columns where the table name already makes it obvious, but keeping it on the two ID columns since they'll be used to join to other tables later and need to stay unambiguous.

In [0]:
# STEP 2: Rename the columns

df_renamed = (
    df_customers_bronze
    .withColumnRenamed("customer_zip_code_prefix","zip_code_prefix")
    .withColumnRenamed("customer_city","city")
    .withColumnRenamed("customer_state","state")
    # customer_id and customer_unique_id aren't changed on purpose
)

display(df_renamed.limit(5))

## Step 3: Clean up city and state values

Raw text often has inconsistent casing or stray spaces. I standardize both so that later grouping — e.g. "sales by state" — doesn't accidentally split one real value into two.

In [0]:
# STEP 3: Clean city and state values

from pyspark.sql.functions import initcap, trim, upper

df_cleaned = (
    df_renamed
    .withColumn("city",initcap(trim("city")))   # "sao paulo " -> "Sao Paulo"
    .withColumn("state",upper(trim("state")))    # " sp"        -> "SP"
)

display(df_cleaned.limit(5))

### Step 4: Check for nulls in the ID columns

customer_id and customer_unique_id should never be null. If either is, that row is broken. 

In [0]:
# STEP 4: Check for nulls in the ID columns

null_id_count = df_cleaned.filter(
    (df_cleaned.customer_id.isNull()) | (df_cleaned.customer_unique_id.isNull())
).count()

print(f"Rows with a null customer_id or customer_unique_id: {null_id_count}")

### Step 5: Validate uniqueness

customer_id should be unique — one row per order-customer link. customer_unique_id is allowed to repeat, so I'm not checking that one for uniqueness, only nulls (already done in Step 4).

In [0]:
# STEP 5: Validate uniqueness

total_rows = df_cleaned.count()
distinct_customer_id = df_cleaned.select("customer_id").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct customer_id per unique order: {distinct_customer_id}")

# These two numbers should match exactly. If they don't, customer_id has duplicates,
# which would break its use as a safe join key later.

### Step 6: Drop _rescued_data

This column only exists to catch malformed rows during the Bronze load, and we already confirmed it was clean (0 rescued) back in Part 4. It's a Bronze diagnostic, not something Silver needs to carry forward.

In [0]:
# STEP 6: Drop _rescued_data

# Drop _rescued_data before writing to Silver
df_cleaned = df_cleaned.drop("_rescued_data")


### Step 7: Write to Silver

_source_file and _ingested_at will be carried forward from Bronze unchanged, I always want to know which Bronze load produced a given Silver row.

In [0]:
# STEP 7: Write to Silver
df_cleaned.write.format("delta").mode("overwrite").saveAsTable("olist_catalog.silver.customers")

print("Written: olist_catalog.silver.customers")

_____________________________________________________________________

# Silver Layer: Sellers

Goal: take the raw `sellers` table from Bronze and make a clean, trustworthy version of it in Silver.

Bronze holds the data exactly as it arrived, every column is text, nothing has been corrected. Silver is where it becomes usable.

Sellers table transformation steps
1. Look at the data first
2. Rename columns where the table name already makes the prefix redundant
3. Clean up city and state text formatting
4. Check for nulls in seller_id
5. Validate seller_id is unique
6. Drop _rescued_data (new — Bronze diagnostic column, not needed in Silver)
7. Write to Silver

## Step 1 for sellers: Look at the data first

Same as customers — before changing anything, I inspect the raw Bronze sellers table so every decision after this is based on what's actually there.

In [0]:
# STEP 1: Look at the data first
df_sellers_bronze = spark.table("olist_catalog.bronze.sellers")

# Quick peek at real values
display(df_sellers_bronze.limit(5))

# Confirm column names and types
df_sellers_bronze.printSchema()

# Row count now, to compare against after every step
print(f"Bronze row count: {df_sellers_bronze.count()}")

### Step 2 for sellers: Rename the columns

Dropping the seller_ prefix where the table name already makes it obvious, but keeping it on seller_id since it's the join key to other tables and needs to stay unambiguous.

In [0]:
# STEP 2: Rename the columns

df_renamed = (
    df_sellers_bronze
    .withColumnRenamed("seller_zip_code_prefix","zip_code_prefix")
    .withColumnRenamed("seller_city","city")
    .withColumnRenamed("seller_state","state")
    # seller_id is left as it is on purpose
)

display(df_renamed.limit(5))

In [0]:
# STEP 3: Clean city and state values

df_cleaned = (
    df_renamed
    .withColumn("city", initcap(trim("city"))) # "sao paulo " -> "Sao Paulo"
    .withColumn("state", upper(trim("state"))) # " sp" -> "SP"
)

display(df_cleaned.limit(5))

###Step 4 for sellers: Check for nulls in seller_id

seller_id should never be null, it's the only identifier this table has.

In [0]:
# STEP 4: Check for nulls in seller_id
null_id_count = df_cleaned.filter(df_cleaned.seller_id.isNull()).count()

print(f"Rows with a null seller_id: {null_id_count}")
# If this is 0, no action needed. If not, I'll decide how to handle those rows explicitly
# rather than dropping them automatically.

### Step 5 for sellers: Validate uniqueness

seller_id should be unique — one row per seller. I check this rather than assume it, since a duplicate would break it as a safe join key later.

In [0]:
# STEP 5: Validate uniqueness
total_rows = df_cleaned.count()
distinct_seller_id = df_cleaned.select("seller_id").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct seller_id: {distinct_seller_id}")
# These two numbers should match exactly. If they don't, seller_id has duplicates.

### Step 6 for sellers: Drop _rescued_data

Same reasoning as customers — this column only exists to catch malformed rows during the Bronze load, and we already confirmed it was clean (0 rescued) back in Part 4. It's a Bronze diagnostic, not something Silver needs to carry forward.

In [0]:
# STEP 6: Drop _rescued_data
df_cleaned = df_cleaned.drop("_rescued_data")

display(df_cleaned.limit(5))

In [0]:
# STEP 7: Write to Silver
df_cleaned.write.format("delta").mode("overwrite").saveAsTable("olist_catalog.silver.sellers")

print("Written: olist_catalog.silver.sellers")

________________________________________________________________________

# Silver Layer: Products

This one's different from the first two, it's not just IDs and location text, it has real numeric/dimensional data, so the planned list is a bit longer:

1. Look at the data first
2. Rename columns (fixed Olist's lenght → length typo)
3. Cast numeric columns (name_length, description_length, photos_qty → integer; weight_g, length_cm, height_cm, width_cm → double)
4. Check for nulls — found 610 genuine gaps (name/description/photos) and 2 (dimensions); kept as null, not dropped or imputed — decided that's a Gold/analyst decision, not Silver's job
5. Validate product_id uniqueness — confirmed (32,951 = 32,951)
6. Drop _rescued_data
7. Write to Silver

### Step 1 for products: Look at the data first

Inspect the raw Bronze products table before changing anything.

In [0]:
# STEP 1: Look at the data first
df_products_bronze = spark.table("olist_catalog.bronze.products")

# Quick peek at real values
display(df_products_bronze.limit(5))

# Confirm column names and types (everything should still be string, carried over from Bronze)
df_products_bronze.printSchema()

# Row count now, to compare against after every step
print(f"Bronze row count: {df_products_bronze.count()}")

### Step 2 for products: Rename the columns

Dropping the product_ prefix where the table name makes it redundant, fixing the "lenght" → "length" typo, and keeping product_id since it's the join key.

In [0]:
# STEP 2: Rename the columns
df_renamed = (
    df_products_bronze
    .withColumnRenamed("product_category_name", "category_name")
    .withColumnRenamed("product_name_lenght", "name_length")        # fixing Olist's spelling
    .withColumnRenamed("product_description_lenght", "description_length")  # same fix
    .withColumnRenamed("product_photos_qty", "photos_qty")
    .withColumnRenamed("product_weight_g", "weight_g")
    .withColumnRenamed("product_length_cm", "length_cm")
    .withColumnRenamed("product_height_cm", "height_cm")
    .withColumnRenamed("product_width_cm", "width_cm")
    # product_id is left as-is on purpose
)

display(df_renamed.limit(5))

### Step 3 for products: Cast numeric columns

These six columns arrived as strings (Bronze's promise: nothing typed until Silver). Now that we're deliberately deciding, photos_qty is a whole number (count of photos), while the rest are measurements that can have decimals.

In [0]:
# STEP 3: Cast numeric columns
from pyspark.sql.functions import col

df_cast = (
    df_renamed
    .withColumn("name_length", col("name_length").cast("integer"))
    .withColumn("description_length", col("description_length").cast("integer"))
    .withColumn("photos_qty", col("photos_qty").cast("integer"))
    .withColumn("weight_g", col("weight_g").cast("double"))
    .withColumn("length_cm", col("length_cm").cast("double"))
    .withColumn("height_cm", col("height_cm").cast("double"))
    .withColumn("width_cm", col("width_cm").cast("double"))
)

df_cast.printSchema()
display(df_cast.limit(5))

### Step 4 for products: Check for nulls

Two different kinds of nulls to check here: nulls in product_id (broken row, same as before), and nulls in the numeric columns — some of which may be genuinely missing in the source, some of which may be new casts-gone-wrong from Step 3. I'll check both separately.

In [0]:
# STEP 4: Check for nulls

# product_id should never be null - same check as customers/sellers
null_product_id = df_cast.filter(df_cast.product_id.isNull()).count()
print(f"Rows with a null product_id: {null_product_id}")

# Check each numeric column separately, so I know exactly which columns have gaps
numeric_cols = ["name_length", "description_length", "photos_qty", "weight_g", "length_cm", "height_cm", "width_cm"]

for c in numeric_cols:
    null_count = df_cast.filter(col(c).isNull()).count()
    print(f"Nulls in {c}: {null_count}")

In [0]:
# Confirm it's the same 610 rows across all three columns, not different rows landing on the same count by coincidence
same_rows_check = df_cast.filter(
    col("name_length").isNull() &
    col("description_length").isNull() &
    col("photos_qty").isNull()
).count()

print(f"Rows null in all three columns together: {same_rows_check}")

### Step 5 for products: Validate uniqueness

product_id should be unique, one row per product.

In [0]:
# STEP 5: Validate uniqueness
total_rows = df_cast.count()
distinct_product_id = df_cast.select("product_id").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct product_id: {distinct_product_id}")

### Step 6 for products: Drop _rescued_data

In [0]:
# STEP 6: Drop _rescued_data
df_cast = df_cast.drop("_rescued_data")

display(df_cast.limit(5))

### Step 7 for products: Write to Silver

Last step for this table. _source_file and _ingested_at carry forward unchanged, same as customers and sellers.

In [0]:
# STEP 7: Write to Silver
df_cast.write.format("delta").mode("overwrite").saveAsTable("olist_catalog.silver.products")

print("Written: olist_catalog.silver.products")

___________________________________________________________________

%md
# Silver Layer: Product_category_translation

This one's small and simple, just two columns, Portuguese category name mapped to English.

1. Look at the data first
2. Rename columns for clarity
3. Check for nulls in either column
4. Validate no duplicate Portuguese category names (each should map to exactly one English name)
5. Drop _rescued_data
6. Write to Silver

### Step 1 for product_category_translation: Look at the data first

Inspect the raw Bronze data before changing anything.

In [0]:
# STEP 1: Look at the data first
df_translation_bronze = spark.table("olist_catalog.bronze.product_category_translation")

# Quick peek at real values
display(df_translation_bronze.limit(5))

# Confirm column names and types
df_translation_bronze.printSchema()

# Row count now, to compare against after every step
print(f"Bronze row count: {df_translation_bronze.count()}")

### Step 2 for product_category_translation: Rename the columns

Both original names are already clear. Shortening them, and matching category_name to the same name we used in products Silver, so the future Gold join lines up cleanly on matching column names.

In [0]:
# STEP 2: Rename the columns
df_renamed = (
    df_translation_bronze
    .withColumnRenamed("product_category_name", "category_name")           # matches products.category_name
    .withColumnRenamed("product_category_name_english", "category_name_english")
)

display(df_renamed.limit(5))

### Step 3 for product_category_translation: Check for nulls

Both columns matter equally here, this table's only purpose is mapping one name to another, so a null in either side breaks that mapping.

In [0]:
# STEP 3: Check for nulls
null_category_name = df_renamed.filter(col("category_name").isNull()).count()
null_category_english = df_renamed.filter(col("category_name_english").isNull()).count()

print(f"Nulls in category_name: {null_category_name}")
print(f"Nulls in category_name_english: {null_category_english}")

### Step 4 for `product_category_translation`: Validate no duplicate Portuguese names

Each Portuguese category name should map to exactly one English name. If `category_name` had duplicates, a future join from `products` to this table could produce more rows than expected, one product matching two translation rows instead of one.

In [0]:
# STEP 4: Validate no duplicate category_name
total_rows = df_renamed.count()
distinct_category_name = df_renamed.select("category_name").distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct category_name: {distinct_category_name}")
# These should match. If they don't, the same Portuguese name maps to more than one English name somewhere.

### Step 5 for product_category_translation: Drop _rescued_data

Same reasoning as every other table Bronze-only diagnostic column, already confirmed clean in Part 4, not needed in Silver.

In [0]:
# STEP 5: Drop _rescued_data
df_renamed = df_renamed.drop("_rescued_data")

display(df_renamed.limit(5))

### Step 6 for product_category_translation: Write to Silver

Last step for this table, and the last table in silver-reference. _source_file and _ingested_at carry forward unchanged, same as every other table.

In [0]:
# STEP 6: Write to Silver
df_renamed.write.format("delta").mode("overwrite").saveAsTable("olist_catalog.silver.product_category_translation")

print("Written: olist_catalog.silver.product_category_translation")

### Final validation pass: silver-reference

One combined check across all four tables -- row counts against Bronze, plus a spot check that each table actually landed in olist_catalog.silver.

In [0]:
# FINAL VALIDATION: silver-reference branch

tables = ["customers", "sellers", "products", "product_category_translation"]

print(f"{'Table':<32} {'Bronze rows':>12} {'Silver rows':>12} {'Match':>8}")
print("-" * 68)

for t in tables:
    bronze_count = spark.table(f"olist_catalog.bronze.{t}").count()
    silver_count = spark.table(f"olist_catalog.silver.{t}").count()
    match = "✅" if bronze_count == silver_count else "❌"
    print(f"{t:<32} {bronze_count:>12} {silver_count:>12} {match:>8}")